## Model Training On Real Data

In [ ]:
# import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Load Dataset

In [ ]:
transactions_data = pd.read_csv("data/credit_card_data_real.zip", compression="zip")
print(transactions_data.head())

## Class Imbalance

As it can be seen here there is a class imbalance poblem with the dataset. The fraudulent transactions are only 492 or less than 1% of the entire dataset

In [ ]:
print(transactions_data["Class"].value_counts())
print()
print(transactions_data["Class"].value_counts(normalize=True))

In [ ]:
plt.bar(transactions_data["Class"].value_counts().index, transactions_data["Class"].value_counts().values, color=["blue", "orange"])
plt.xticks([0, 1], ["Legit", "Fraud"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Target Labels Distribution")
plt.show()

## Data Preparation

### Data Resampling

As seen before the data has to be resampled to mitigate the class imbalance problem. SMOTE is used again to over-sample the minority class

In [ ]:
from imblearn.over_sampling import SMOTE

X = transactions_data.drop(["Class"], axis=1)
y = transactions_data["Class"]

smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
plt.bar(y_resampled.value_counts().index, y_resampled.value_counts().values, color=["blue", "orange"])
plt.xticks([0, 1], ["Legit", "Fraud"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Target Labels Distribution")
plt.show()

## Model Building

It's time to try a few machine learning models and see how they perform on real data.

In [ ]:
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

### Evaluation Metrics

Given the imbalnced dataset, some suitable metrics are __ROC AUC__ score, __average precision__ score and __f1__ score.

- **ROC AUC**: is the area under the Receiver Operating Characteristic curve

- **average precision**: summarizes a precision-recall curve as the weighted mean of precisions achieved at each threshold, with the increase in recall from the previous threshold used as the weight:

$$ AP = \sum(R_n - R_{n - 1})P_n $$

  where $P_n$ and $ R_n $ are the precision and recall at the nth threshold 

- **f1 score**: is the harmonic mean of precision and recall
$$ F1 = \frac{2 * TP}{2 * TP + FP + FN} $$

### Logistic Regression

Bellow a pipeline was created with StandardScaler, SMOTE over-sampling and a LogisticRegression model. The model was trained using stratified cross-validation and achived a roc_aus score of 98%, average precision score of 74% and f1 score of 11%

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate

# Define components 
scaler = StandardScaler()
log_reg = LogisticRegression(random_state=42, class_weight="balanced")

# Define pipeline
log_reg_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", log_reg)
    ]
)

# Cross-validation Iterator
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Metrics
scoring = {
    "roc_aus": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1"
}

# Run cv
results = cross_validate(
    log_reg_pipeline,
    X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    return_estimator=True,
    return_indices=True
)

for metric in scoring:
    print(f"{metric}: {results["test_" + metric].mean():.4f}")

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_cv_results(results, X, y)
plt.show()

### Stochastic Gradient Descent Classifier

Bellow a pipeline was created with StandardScaler, SMOTE over-sampling and a SGDClassifier model. The model was trained using stratified cross-validation and achived a roc_aus score of 98%, average precision score of 73% and f1 score of 10%

In [ ]:
sgd_clf = SGDClassifier(random_state=42, class_weight="balanced")

sgd_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", sgd_clf)
    ]
)

results = cross_validate (
    sgd_pipeline,
    X, y, 
    scoring=scoring,
    cv=cv,
    return_train_score=False,
    return_estimator=True,
    return_indices=True
)

for metric in scoring:
    print(f"{metric}: {results["test_" + metric].mean():.4f}")

In [ ]:
RocCurveDisplay.from_cv_results(results, X, y)
plt.show()

### Gradient Boosting Classifier

The Gradient Boosting Classifier pipieline achived a roc_auc score of 97%, 83% average precision score and 66% f1 score. This is the best model so far although it takes longer to train

In [ ]:
gb_clf = HistGradientBoostingClassifier(random_state=42)

gb_clf_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", gb_clf)
    ]
)

results = cross_validate(
    gb_clf_pipeline,
    X, y,
    scoring=scoring,
    cv=cv,
    return_train_score=False,
    return_estimator=True,
    return_indices=True
)

for metric in scoring:
    print(f"{metric}: {results["test_" + metric].mean():.4f}")

In [ ]:
RocCurveDisplay.from_cv_results(results, X, y)
plt.show()

## Hyper-parameter Tuning

### Logistic Regression

After hyper-parameter tuning, the best LogisticRegression model achived an average precision score of 75% which is slightly better than before

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__C": [0.001, 0.01, 0.1, 1, 10],
    "model__fit_intercept": [False, True],
    "model__class_weight": [None, "balanced"]
}

rand_search = RandomizedSearchCV(
    log_reg_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

rand_search.fit(X, y)

In [ ]:
best_model = rand_search.best_estimator_
best_params = rand_search.best_params_
best_score = rand_search.best_score_

print(f"Best hyper-parameters: {best_params}")
print(f"Best score: {best_score:.4f}")

### Stochastic Gradient Descent Classifier

The best Stochastic Gradient Descent model achived an average precision score of 74% which % lower the the best Logistic Regrssion model.

In [ ]:
param_dist = {
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__alpha": [0.00001, 0.0001, 0.001, 0.01, 0.1, 1],
    "model__loss": ["hinge", "log_loss"],
}

rand_search = RandomizedSearchCV(
    sgd_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

rand_search.fit(X, y)

In [ ]:
best_model = rand_search.best_estimator_
best_params = rand_search.best_params_
best_score = rand_search.best_score_

print(f"Best hyper-parameters: {best_params}")
print(f"Best score: {best_score:.4f}")

### Gradient Boosting Classifier

The best Gradient Bosting Classifier achived an average precision similar to the best Logistic Regression model but slightly lower than the Gradient Boosting Classifier with default hyper-parameters. Additionally this model takes a lot of time to tune.

In [ ]:
param_dist = {
    "smote__k_neighbors": [3, 5, 7],
    "smote__sampling_strategy": [0.2, 0.5, 1.0],
    "model__learning_rate": np.arange(0.01, 0.02, 0.01),
    "model__max_depth": [3, 5, 7, 9]
}

rand_search = RandomizedSearchCV(
    gb_clf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

rand_search.fit(X, y)

In [ ]:
best_model = rand_search.best_estimator_
best_params = rand_search.best_params_
best_score = rand_search.best_score_

print(f"Best hyper-parameters: {best_params}")
print(f"Best score: {best_score:.4f}")

## Final Evaluation

The best model so far seems to be the Gradient Bossting Classifier. Bellow the model is evaluated on unseen data and achived a f1 score of 61%, average precision score of 41% and roc auc score of 96%.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

gb_clf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = gb_clf_pipeline.predict(X_test)
y_pred_proba = gb_clf_pipeline.predict_proba(X_test)[:, 1]

print(f"Test set f1 score:{f1_score(y_test, y_pred):.4f}")
print(f"Test set average precision score: {average_precision_score(y_test, y_pred):.4f}")
print(f"Test set roc auc score: {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_pred_proba, plot_chance_level=True)
plt.show()

As seen from the confusion matrix bellow, the model corectly classifed 85 smaples from the test set as fraudulent and 56772 samples as legit transctions. The false negatives are an acceptable amount (13) but there are a lot of false positives (92).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

## Tuning Classifiation Threshold

The optimal classifiction threshold is 98%. This means that if a transction has greater than 98% probability of being fraudulent it will be classified as fraud. The thresholded Gradient Boosting Classifier acived a f1 score of 85%, an average precision score of 73% and a roc auc score of 96%. 

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV

calibrated_clf = TunedThresholdClassifierCV(
    gb_clf_pipeline, 
    cv=cv,
    scoring="average_precision",
    n_jobs=-1,
    random_state=42
)

calibrated_clf.fit(X_train, y_train)

In [ ]:
y_pred = calibrated_clf.predict(X_test)
y_pred_proba = calibrated_clf.predict_proba(X_test)[:, 1]

print(f"Cut-off point found at {calibrated_clf.best_threshold_:.3f}")
print(f"Test set f1 score: {f1_score(y_test, y_pred):.4f}")
print(f"Test set average precision score: {average_precision_score(y_test, y_pred):.4f}")
print(f"Test set roc auc score: {roc_auc_score(y_test, y_pred_proba)}")

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_pred_proba, plot_chance_level=True)
plt.show()

As seen from the confusion matrix bellow the thresholded classifer makes less false positive classifications but the higher threshold also introduced more false negatives.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()